# Processamento de Língua Natural - EP (Testes)
Baseline: Regressão Logística com contagem TF-IDF

## Bibliotecas
Sklearn, Pandas

In [46]:
import pandas as pd
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

## Importação do Conjunto de Treino
O Pandas importa ambos os conjuntos de dados (treino e teste) como strings, para evitar erros de tipagem no momento da tokenização.

In [47]:
dados_treino = pd.read_excel("dados/train.xlsx", converters={'resp_text':str,'clarity':str})

X_treino, Y_treino = dados_treino["resp_text"].tolist(), dados_treino["clarity"].tolist()

In [48]:
dados_treino

,resp_text,clarity
0,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c5
1,"Prezada cidadã, As informações sobre óbitos ...",c1
2,"Prezado Senhor Julio, A Ouvidoria-Geral da P...",c1
3,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c234
4,"Senhor, O Serviço de Informações ao Cidadão d...",c234
...,...,...
20087,Prezado Sr. Luciano Wolff A sua solicitação ...,c1
20088,"PREZADO CIDADãO, A COORDENAçãO DO PROGRAMA FA...",c1
20089,"Prezado, Informamos que o Ministério da Agr...",c1
20090,"Prezado Senhor, Encaminhamos em anexo respos...",c1


## Tokenização (TF-IDF de Palavras + Caracteres)

In [49]:
# Combinação de TF-IDF de palavras e caracteres
vect = FeatureUnion([
    (
        'word',
        TfidfVectorizer(
            analyzer='word',
            ngram_range=(1, 2),
            min_df=10,
            max_df=0.95,
            max_features=20000,
            sublinear_tf=True,
            strip_accents='unicode'
        )
    ),
    (
        'char',
        TfidfVectorizer(
            analyzer='char_wb',
            ngram_range=(3, 5),
            min_df=10,
            max_features=20000,
            sublinear_tf=True
        )
    )
])

## Classificador (Regressão Logística)

In [50]:
clf = LogisticRegression(
    C=0.5,
    class_weight='balanced',
    max_iter=150,
    solver='saga',
    tol=1e-3,
    random_state=42
)

## Validação Cruzada

In [51]:
pipeline = Pipeline([
    ('tfidf', vect),
    ('clf', clf)
])

cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

resultados = cross_validate(
    pipeline,
    X_treino,
    Y_treino,
    scoring='accuracy',
    cv=cv,
    return_train_score=True
)

acuracia_validacao = resultados['test_score'].mean()
acuracia_treino_cv = resultados['train_score'].mean()
gap = acuracia_treino_cv - acuracia_validacao

print(f"Acurácia Média da Validação Cruzada: {acuracia_validacao:.2%}")
print(f"Acurácia Média de Treino nos Folds: {acuracia_treino_cv:.2%}")
print(f"Gap Treino-Validação: {gap:.2%}")

Acurácia Média da Validação Cruzada: 46.22%
Acurácia Média de Treino nos Folds: 67.16%
Gap Treino-Validação: 20.94%


## Treino do Modelo

In [52]:
pipeline.fit(X_treino, Y_treino)
print(f"Acurácia do Treino (modelo final): {pipeline.score(X_treino, Y_treino):.2%}")

Acurácia do Treino (modelo final): 66.65%


## Execução do Teste e Salvamento do Resultado

In [53]:
# Importação dos dados de teste
dados_teste = pd.read_excel("dados/test1.xlsx", converters={'resp_text': str, 'clarity': str})
X_teste = dados_teste["resp_text"].tolist()

# Aplicação do modelo no teste
Y_previsto = pipeline.predict(X_teste)

In [54]:
teste_rotulado = pd.DataFrame({
    'resp_text': X_teste,
    'clarity': Y_previsto
})

teste_rotulado.to_excel("teste_rotulado.xlsx", index=False)